# Market Risk Engine — Deep Dive

A guided tour of the `market-risk-engine` library: VaR across methods, Expected
Shortfall, Basel backtesting, stress testing, risk decomposition, and capital
allocation, applied to the bundled sample equity portfolio.

> Run from the repository root so the top-level packages import correctly.
> The bundled data is **synthetic** and for demonstration only.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from data import equity_portfolio
portfolio = equity_portfolio(notional=10_000_000)
print(portfolio.n_assets, "assets:", portfolio.asset_names)
print("returns:", portfolio.returns.shape,
      portfolio.returns.index.min().date(), "->", portfolio.returns.index.max().date())
portfolio.returns.head()

## 1. Portfolio overview

Annualised statistics and the fat-tailed, mildly-skewed return profile that
makes parametric-Normal VaR understate tail risk.

In [ ]:
from utils.statistics import summary_statistics, sharpe_ratio, max_drawdown
r = portfolio.return_series()
s = summary_statistics(r)
print(f"Ann. vol        {s['std']*np.sqrt(252):.2%}")
print(f"Skewness        {s['skewness']:.3f}")
print(f"Excess kurtosis {s['excess_kurtosis']:.2f}")
print(f"Sharpe          {sharpe_ratio(r):.2f}")
print(f"Max drawdown    {max_drawdown(r):.2%}")
print(f"Jarque-Bera p   {s['jb_pvalue']:.2e}  (reject normality)")

## 2. Value-at-Risk across methods

Every estimator at 99% / 1-day. Note how Historical and Student-t VaR exceed
parametric-Normal because they capture the fat left tail.

In [ ]:
from var import VaREngine
results = VaREngine().compute_all(portfolio, confidence=0.99, horizon_days=1)
pd.DataFrame([{"Method": x.method, "VaR ($)": x.var, "VaR (%)": x.var_pct}
              for x in results.values()]).sort_values("VaR ($)")

In [ ]:
# Unified VaR & ES comparison table
VaREngine().comparison_table(portfolio, confidence=0.99)

## 3. Expected Shortfall

ES averages the losses *beyond* VaR. The ES/VaR ratio shows the extra capital a
coherent tail measure demands; for the Normal distribution at 99% it is ≈1.15.

In [ ]:
from expected_shortfall import ESEngine
es = ESEngine().compute_all(portfolio, confidence=0.99)
pd.DataFrame([{"Method": x.method, "VaR ($)": x.var, "ES ($)": x.es,
               "ES/VaR": x.es_var_ratio} for x in es.values()])

## 4. Basel backtesting & traffic light

A rolling out-of-sample backtest with Kupiec, Christoffersen and the Basel
traffic-light zone / capital multiplier.

In [ ]:
from backtesting import BacktestEngine
result, frame = BacktestEngine().run(portfolio, "historical", confidence=0.99,
                                     test_window=500, estimation_window=500)
print(result)
BacktestEngine().run_all_methods(portfolio, confidence=0.99, test_window=500)

## 5. Stress testing

Replay historical crises that fall inside the sample window (2015 China, 2020
COVID, 2022 bear), then a hypothetical uniform shock.

In [ ]:
from stress_testing import HistoricalScenarioEngine, HypotheticalScenarioEngine
HistoricalScenarioEngine().run_all(portfolio)

In [ ]:
# Shock every asset by -10% -> P&L equals -10% of notional (weights sum to 1)
sc = HypotheticalScenarioEngine().apply_custom_shock(
    portfolio, {a: -0.10 for a in portfolio.asset_names})
print(sc)

## 6. Risk decomposition

Component VaR is **exactly additive** (sums to total VaR); PCA shows how few
factors drive most of the portfolio variance.

In [ ]:
from risk_decomposition import ComponentVaR, FactorRiskDecomposition
cv = ComponentVaR().compute(portfolio, confidence=0.99, cov_method="sample")
print("Sum of Component VaR:", cv["ComponentVaR($)"].sum())
cv

In [ ]:
pca = FactorRiskDecomposition().pca_decomposition(portfolio, n_factors=5)
pd.DataFrame({"Variance explained": pca["variance_explained"][:5],
              "Cumulative": np.cumsum(pca["variance_explained"])[:5]},
             index=[f"PC{i+1}" for i in range(5)])

## 7. Capital allocation

Economic capital at 99.9% over a 1-year horizon, and the Basel III vs Basel IV
(FRTB) capital comparison.

In [ ]:
from capital import EconomicCapitalEngine, BaselCapitalEngine
eco = EconomicCapitalEngine()
print(f"Economic capital (99.9%, 1Y): ${eco.economic_capital(portfolio):,.0f}")
print(f"Diversification benefit:      {eco.diversification_benefit(portfolio):.1%}")
BaselCapitalEngine().capital_comparison(portfolio)

## 8. Model analytics

GARCH(1,1) volatility persistence and an EVT (peaks-over-threshold) tail fit.

In [ ]:
from models import GARCHModel
from diagnostics import TailAnalyzer
g = GARCHModel(); params = g.fit(portfolio.return_series())
print(f"GARCH persistence (alpha+beta): {params.persistence:.3f}")
ta = TailAnalyzer(); gpd = ta.fit_gpd(portfolio.return_series(), threshold_quantile=0.95)
print(f"EVT tail index xi: {gpd['xi']:.3f}  (xi>0 => heavy tail)")
print(f"EVT VaR(99%):      {ta.evt_var(gpd['xi'], gpd['beta'], gpd['threshold'], int(gpd['n_total']), int(gpd['n_exceedances']), 0.99):.2%}")

---
### Next steps

* Launch the full interactive dashboard: `streamlit run app.py`
* Generate the research paper: `python paper/generate_paper.py`
* Swap in your own returns CSV via `data.MarketDataLoader.load_returns_csv`.